In [ ]:
# =========================================================
# IMAGE SEGMENTATION USING:
# 1. OTSU THRESHOLDING
# 2. K-MEANS CLUSTERING
#
# EVALUATION METRICS:
# 1. MSE
# 2. MAE
# 3. RMSE
# 4. IoU (Jaccard Index)
# 5. Dice Coefficient
# 6. Accuracy
# 7. Precision
# 8. Recall
# 9. F1-Score
# 10. Euclidean Distance
# 11. Manhattan Distance
# =========================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =========================================================
# LOAD IMAGE
# =========================================================

img = cv2.imread(
    "./data/002.Laysan_Albatross/Laysan_Albatross_0050_870.jpg"
)

# Ground Truth Segmentation Mask
gt = cv2.imread(
    "./segmentations/002.Laysan_Albatross/Laysan_Albatross_0050_870.png",
    0
)

# Convert image BGR -> RGB
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

# =========================================================
# CONVERT GROUND TRUTH TO BINARY
# =========================================================

_, gt = cv2.threshold(gt, 127, 255, cv2.THRESH_BINARY)

# =========================================================
# 1. OTSU SEGMENTATION
# =========================================================

# OTSU automatically finds best threshold value
_, otsu = cv2.threshold(
    gray,
    0,
    255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

# =========================================================
# 2. K-MEANS SEGMENTATION
# =========================================================

# Reshape image into pixel list
pixels = img.reshape((-1, 3))

# Convert to float32
pixels = np.float32(pixels)

# Create KMeans model
kmeans = KMeans(
    n_clusters=2,
    random_state=0,
    n_init=10
)

# Predict cluster labels
labels = kmeans.fit_predict(pixels)

# Cluster centers
centers = np.uint8(kmeans.cluster_centers_)

# Recreate segmented image
segmented = centers[labels.flatten()]
kseg = segmented.reshape(img.shape)

# Convert segmented image to grayscale
kgray = cv2.cvtColor(kseg, cv2.COLOR_RGB2GRAY)

# Convert to binary mask
_, kmeans_bin = cv2.threshold(
    kgray,
    127,
    255,
    cv2.THRESH_BINARY_INV
)

# =========================================================
# METRIC FUNCTION
# =========================================================

def segmentation_metrics(gt, pred):

    # -----------------------------------------------------
    # Flatten arrays
    # -----------------------------------------------------
    gt_flat = gt.flatten()
    pred_flat = pred.flatten()

    # Convert masks to binary (0 and 1)
    gt_bin = (gt_flat > 0).astype(np.uint8)
    pred_bin = (pred_flat > 0).astype(np.uint8)

    # =====================================================
    # 1. MSE
    # =====================================================
    mse = mean_squared_error(gt_flat, pred_flat)

    # =====================================================
    # 2. MAE
    # =====================================================
    mae = mean_absolute_error(gt_flat, pred_flat)

    # =====================================================
    # 3. RMSE
    # =====================================================
    rmse = np.sqrt(mse)

    # =====================================================
    # 4. IoU (Intersection over Union)
    # =====================================================
    intersection = np.logical_and(gt_bin, pred_bin)
    union = np.logical_or(gt_bin, pred_bin)

    iou = np.sum(intersection) / (np.sum(union) + 1e-8)

    # =====================================================
    # 5. Dice Coefficient
    # =====================================================
    dice = (
        2 * np.sum(intersection)
    ) / (
        np.sum(gt_bin) + np.sum(pred_bin) + 1e-8
    )

    # =====================================================
    # 6. Accuracy
    # =====================================================
    accuracy = accuracy_score(gt_bin, pred_bin)

    # =====================================================
    # 7. Precision
    # =====================================================
    precision = precision_score(
        gt_bin,
        pred_bin,
        zero_division=0
    )

    # =====================================================
    # 8. Recall
    # =====================================================
    recall = recall_score(
        gt_bin,
        pred_bin,
        zero_division=0
    )

    # =====================================================
    # 9. F1 Score
    # =====================================================
    f1 = f1_score(
        gt_bin,
        pred_bin,
        zero_division=0
    )

    # =====================================================
    # 10. Euclidean Distance
    # =====================================================
    euclidean = np.linalg.norm(
        gt_flat.astype(np.float32)
        -
        pred_flat.astype(np.float32)
    )

    # =====================================================
    # 11. Manhattan Distance
    # =====================================================
    manhattan = np.sum(
        np.abs(
            gt_flat.astype(np.float32)
            -
            pred_flat.astype(np.float32)
        )
    )

    return {
        "MSE": mse,
        "MAE": mae,
        "RMSE": rmse,
        "IoU": iou,
        "Dice": dice,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Euclidean Distance": euclidean,
        "Manhattan Distance": manhattan
    }

# =========================================================
# CALCULATE METRICS
# =========================================================

otsu_metrics = segmentation_metrics(gt, otsu)

kmeans_metrics = segmentation_metrics(gt, kmeans_bin)

# =========================================================
# PRINT RESULTS
# =========================================================

print("\n==============================")
print("OTSU SEGMENTATION RESULTS")
print("==============================")

for key, value in otsu_metrics.items():
    print(f"{key:20}: {value}")

print("\n==============================")
print("K-MEANS SEGMENTATION RESULTS")
print("==============================")

for key, value in kmeans_metrics.items():
    print(f"{key:20}: {value}")

# =========================================================
# DISPLAY RESULTS
# =========================================================

plt.figure(figsize=(16, 8))

# ---------------------------------------------------------
# Original Image
# ---------------------------------------------------------
plt.subplot(1, 4, 1)

plt.imshow(img)

plt.title("Original Image")

plt.axis("off")

# ---------------------------------------------------------
# Ground Truth
# ---------------------------------------------------------
plt.subplot(1, 4, 2)

plt.imshow(gt, cmap='gray')

plt.title("Ground Truth")

plt.axis("off")

# ---------------------------------------------------------
# OTSU Result
# ---------------------------------------------------------
plt.subplot(1, 4, 3)

plt.imshow(otsu, cmap='gray')

plt.title("OTSU Segmentation")

plt.axis("off")

# ---------------------------------------------------------
# K-Means Result
# ---------------------------------------------------------
plt.subplot(1, 4, 4)

plt.imshow(kmeans_bin, cmap='gray')

plt.title("K-Means Segmentation")

plt.axis("off")

plt.tight_layout()

plt.show()